# Regression Tree

---

## Overview

A **regression tree** applies the same recursive binary splitting as a decision tree classifier, but uses **Mean Squared Error (MSE)** as the splitting criterion instead of Gini impurity.

At each node, we choose the split $(j, t)$ that minimizes:

$$\text{MSE}_{\text{split}} = \frac{N_L}{N} \cdot \text{Var}(y_{\text{left}}) + \frac{N_R}{N} \cdot \text{Var}(y_{\text{right}})$$

Each **leaf node** predicts the **mean** of all target values in that region:

$$\hat{y} = \frac{1}{|\text{leaf}|} \sum_{i \in \text{leaf}} y^{(i)}$$

---

**Dataset:** Gym Members Exercise Tracking (`gym_members_exercise_tracking.csv`)  
**Task:** Predict `Session_Duration (hours)` from other features.
except FileNotFoundError:
    from sklearn.datasets import make_regression
    X, y = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
    print('CSV not found. using synthetic regression data')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning import DecisionTreeRegressor
from rice_ml.preprocess import StandardScaler, train_test_split
from rice_ml.metrics import mse, rmse, r2_score

In [ ]:
try:
    df = pd.read_csv('../../../data/gym_members_exercise_tracking.csv')
    target_col = 'Session_Duration (hours)'
    feature_cols = ['Age', 'Weight (kg)', 'Calories_Burned', 'Workout_Frequency (days/week)', 'Fat_Percentage']
    df = df[feature_cols + [target_col]].dropna()
    X = df[feature_cols].values.astype(float)
    y = df[target_col].values.astype(float)
    print(f'Loaded gym dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import make_regression
    X, y = make_regression(n_samples=300, n_features=5, noise=0.5, random_state=0)
    print('CSV not found. using synthetic regression data')

print(f'Target range: [{y.min():.2f}, {y.max():.2f}]')

In [ ]:
# EDA: scatter each feature vs target
try:
    feat_names = feature_cols
except NameError:
    feat_names = [f'Feature {i}' for i in range(X.shape[1])]

n_features = min(X.shape[1], 4)
fig, axes = plt.subplots(1, n_features, figsize=(4 * n_features, 4))
if n_features == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.scatter(X[:, i], y, alpha=0.4, color='steelblue', s=20)
    ax.set_xlabel(feat_names[i], fontsize=11)
    ax.set_ylabel('Target', fontsize=11)
    ax.set_title(feat_names[i], fontsize=12)
plt.suptitle('Feature vs Target Scatter', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

## Effect of `max_depth` on Regression Error

Shallow trees underfit (high bias), deep trees overfit (low training error but high test error).

In [ ]:
depths = range(1, 12)
train_r2, test_r2 = [], []

for d in depths:
    reg = DecisionTreeRegressor(max_depth=d)
    reg.fit(X_train, y_train)
    train_r2.append(r2_score(y_train, reg.predict(X_train)))
    test_r2.append(r2_score(y_test, reg.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_r2, marker='o', label='Train $R^2$', color='steelblue')
plt.plot(depths, test_r2, marker='s', label='Test $R^2$', color='salmon')
plt.xlabel('max_depth', fontsize=15)
plt.ylabel('$R^2$ Score', fontsize=15)
plt.title('Regression Tree: Depth vs $R^2$', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
best_depth = depths[np.argmax(test_r2)]
reg = DecisionTreeRegressor(max_depth=best_depth)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print(f'Best depth: {best_depth}')
print(f'Test MSE:  {mse(y_test, y_pred):.4f}')
print(f'Test RMSE: {rmse(y_test, y_pred):.4f}')
print(f'Test R²:   {r2_score(y_test, y_pred):.4f}')

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', label='Perfect fit')
plt.xlabel('Actual', fontsize=15)
plt.ylabel('Predicted', fontsize=15)
plt.title(f'Regression Tree (depth={best_depth}): Actual vs Predicted', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
# Residual analysis
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred, residuals, alpha=0.6, color='steelblue', edgecolors='white', s=40)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted', fontsize=13)
axes[0].set_ylabel('Residual', fontsize=13)
axes[0].set_title('Residuals vs Predicted', fontsize=14)

axes[1].hist(residuals, bins=25, color='salmon', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual', fontsize=13)
axes[1].set_ylabel('Count', fontsize=13)
axes[1].set_title('Residual Distribution', fontsize=14)

plt.suptitle('Residual Analysis', fontsize=15)
plt.tight_layout()
plt.show()

print(f'Mean residual:    {residuals.mean():.4f}  (should be near 0)')
print(f'Std of residuals: {residuals.std():.4f}')


## Interpretation

- The regression tree creates **piecewise constant** predictions. each leaf returns a constant mean value.
- Unlike linear regression, it can model **nonlinear relationships** without feature engineering.
- Too deep a tree memorizes training data (each leaf has only 1 point), reducing test $R^2$.